# Run the five-record human-reviewed pilot in Colab

Commit/push the current code before running this notebook. The reviewed gold is
private and excluded from Git: upload `halueval_reviewed_pilot_inputs.zip` separately.
Add your API key in Colab **Secrets** as `GROQ_API_KEY`, granting this notebook access.

This profile uses a streaming **FTS5 lexical index** for pipeline+B0; the existing
BM25 profiles are unchanged. Report this retrieval method with pilot results.
No GPU is needed for prompted verification/repair. Use Python 3.12 for the pinned
annotation environment. Indexing the full FEVER dump needs sufficient local and
Drive storage. No gold evidence is fed to generation.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.12"], check=True)


In [ ]:
from pathlib import Path
import sys, os, subprocess, json, shutil, zipfile
REPO_URL = "https://github.com/harshitap1305/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers.git"
CODE_REF = ""  # Prefer the exact commit you pushed; empty uses the cloned branch.
REPO = Path("/content/hallucination_project")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
if CODE_REF:
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_REF], check=True)
assert (REPO / "requirements-evaluation.txt").is_file(), "Push the current code, then clone that revision."
# Create a dedicated Python 3.12 environment
ENV = Path("/content/reviewed_pilot_env")

PYTHON = str(ENV / "bin/python")
INSTALL_DEPENDENCIES = not Path(PYTHON).exists()
if INSTALL_DEPENDENCIES:
    subprocess.run(["uv", "venv", "--python", "3.12", "--seed", str(ENV)], check=True)

# Verify the isolated interpreter; the outer Colab kernel may be Python 3.13.
assert subprocess.check_output([PYTHON, "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"], text=True).strip() == "3.12", "Select a Python 3.12 environment."
# Verify Python version
subprocess.run([PYTHON, "--version"], check=True)
if INSTALL_DEPENDENCIES:
    subprocess.run([PYTHON, "-m", "pip", "install", "-r", str(REPO / "requirements-evaluation.txt")], check=True)
subprocess.run([PYTHON, "-m", "pip", "check"], check=True)
os.chdir(REPO)
print("Code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# Optional: apply the prepared code ZIP when these changes are not pushed yet.
APPLY_CODE_UPDATE = False  # Set True once to upload pilot_followup_v3_code.zip.
if APPLY_CODE_UPDATE:
    from google.colab import files
    import hashlib, io
    code_uploads = files.upload()
    assert len(code_uploads) == 1, "Select exactly the code update ZIP."
    with zipfile.ZipFile(io.BytesIO(next(iter(code_uploads.values())))) as code_archive:
        manifest = json.loads(code_archive.read("source_manifest.json"))
        names = set(manifest["files"])
        assert set(code_archive.namelist()) == names | {"source_manifest.json"}
        payloads = {}
        for name in names:
            relative = Path(name)
            assert not relative.is_absolute() and ".." not in relative.parts
            assert (relative.parts[0] in {"src", "scripts", "configs", "docs", "notebooks"}
                    or relative.name.startswith("requirements")), "Unexpected code file."
            target = REPO / relative
            assert target.resolve().is_relative_to(REPO.resolve())
            data = code_archive.read(name)
            assert hashlib.sha256(data).hexdigest() == manifest["files"][name]
            payloads[target] = data
        for target, data in payloads.items():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
    print("Code update verified and applied; generating source will be snapshotted in the run.")


## Upload reviewed annotations
Upload the generated ZIP. This copies the five reviewed files without displaying
answers, citations, or credentials in the notebook output.

In [ ]:
from google.colab import files, userdata
uploads = {} if (REPO / "data/gold/pilot_dev.manifest.json").exists() else files.upload()
archives = [name for name in uploads if name.lower().endswith(".zip")]
assert len(archives) <= 1, "Upload exactly the reviewed-input ZIP."
import io
required = {"pilot_dev.jsonl", "pilot_dev.sources.json", "pilot_dev.manifest.json", "pilot_splits.json", "pilot_annotation_review.md"}
GOLD = REPO / "data/gold"
GOLD.mkdir(parents=True, exist_ok=True)
if archives:
    with zipfile.ZipFile(io.BytesIO(uploads[archives[0]])) as archive:
        assert set(archive.namelist()) == required, "Unexpected reviewed-input archive contents."
        for name in sorted(required):
            data = archive.read(name)
            target = GOLD / name
            if target.exists():
                assert target.read_bytes() == data, "Existing gold differs; use a new checkout/version."
            else:
                target.write_bytes(data)
import hashlib
review_manifest = json.loads((GOLD / "pilot_dev.manifest.json").read_text())
for name, expected_hash in review_manifest["files_sha256"].items():
    assert hashlib.sha256((GOLD / name).read_bytes()).hexdigest() == expected_hash, "Reviewed packet checksum mismatch"
# Secrets are inherited by child processes but never printed or written to code.
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
assert os.environ["GROQ_API_KEY"], "Add GROQ_API_KEY to Colab Secrets."
DRIVE_ROOT = Path("/content/hallucination_reviewed_pilot")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

## Build or reuse the full FEVER FTS5 index
The index is built by streaming pages into SQLite, without retaining a BM25
corpus in RAM. Keep `MAX_PASSAGES=None` for the full corpus. A passage cap produces
only a diagnostic subset and must be recorded as such. Existing partial builds
are not automatically overwritten; inspect them before rebuilding.

The downloader uses the current FEVER URL and the authors’ Zenodo mirror, validates
the published MD5 and ZIP before extraction, and records the source and SHA-256.


In [ ]:
import urllib.request
import urllib.error
import hashlib
import json

# Same June 2017 FEVER archive, with the authors' Zenodo deposit as fallback.
# Published checksum: https://zenodo.org/records/4925954
FEVER_WIKI_URLS = (
    "https://fever.ai/download/fever/wiki-pages.zip",
    "https://zenodo.org/records/4925954/files/wiki-pages.zip?download=1",
)
FEVER_WIKI_MD5 = "ed8bfd894a2c47045dca61f0c8dc4c07"


def archive_digests(path):
    md5, sha256 = hashlib.md5(), hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            md5.update(block)
            sha256.update(block)
    return md5.hexdigest(), sha256.hexdigest()


def download_fever_archive(archive, urls=FEVER_WIKI_URLS, expected_md5=FEVER_WIKI_MD5):
    manifest_path = archive.with_suffix(".download.json")
    if archive.exists():
        md5, sha256 = archive_digests(archive)
        if md5 != expected_md5 or not zipfile.is_zipfile(archive):
            raise ValueError(f"Cached archive is invalid: {archive}. Move it aside and rerun this cell.")
        # Validate an existing complete cache without downloading it again.
        previous = {}
        if manifest_path.exists():
            try:
                previous = json.loads(manifest_path.read_text())
            except (ValueError, OSError):
                previous = {}
        source = previous.get("source", "existing_verified_cache") if previous.get("sha256") == sha256 else "existing_verified_cache"
        manifest_path.write_text(json.dumps({"source": source, "md5": md5,
            "sha256": sha256, "bytes": archive.stat().st_size}, indent=2))
        return archive
    failures = []
    part = archive.with_suffix(archive.suffix + ".part")
    for url in urls:
        try:
            print("Downloading FEVER archive:", url)
            request = urllib.request.Request(url, headers={"User-Agent": "FEVER-research-pilot/1.0"})
            with urllib.request.urlopen(request, timeout=90) as response, part.open("wb") as stream:
                size = response.headers.get("Content-Length")
                downloaded, reported = 0, 0
                for block in iter(lambda: response.read(1024 * 1024), b""):
                    stream.write(block)
                    downloaded += len(block)
                    if downloaded - reported >= 64 * 1024 * 1024:
                        print(f"  {downloaded / 1024**2:.0f} MiB downloaded")
                        reported = downloaded
                if size is not None and downloaded != int(size):
                    raise ValueError("Incomplete FEVER archive download")
            md5, sha256 = archive_digests(part)
            if md5 != expected_md5 or not zipfile.is_zipfile(part):
                raise ValueError("FEVER archive checksum/ZIP validation failed")
            part.replace(archive)
            manifest_path.write_text(json.dumps({"source": url, "md5": md5,
                "sha256": sha256, "bytes": downloaded}, indent=2))
            print("FEVER archive verified.")
            return archive
        except (urllib.error.URLError, OSError, ValueError) as exc:
            # Disk/permission failures cannot be fixed by switching download hosts.
            if isinstance(exc, OSError) and exc.errno in (13, 28, 30):
                raise
            failures.append(f"{url}: {type(exc).__name__}: {exc}")
            print("Download source failed; trying the next published source.")
    raise RuntimeError("Could not download a verified FEVER archive:\n" + "\n".join(failures))

INDEX = REPO / "indexes/fever_fts.sqlite"
DRIVE_INDEX = DRIVE_ROOT / "fever_fts.sqlite"
MAX_PASSAGES = None
INDEX.parent.mkdir(parents=True, exist_ok=True)
if not INDEX.exists() and DRIVE_INDEX.exists():
    shutil.copyfile(DRIVE_INDEX, INDEX)
if not INDEX.exists():
    archive = Path("/content/fever-wiki-pages.zip")
    download_fever_archive(archive)
    extract_root = Path("/content/fever_wiki")
    wiki = extract_root / "wiki-pages"
    extraction_marker = extract_root / ".extraction_complete"
    archive_sha256 = json.loads(archive.with_suffix(".download.json").read_text())["sha256"]
    if not extraction_marker.exists() or extraction_marker.read_text() != archive_sha256:
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as zipped:
            for member in zipped.infolist():
                destination = (extract_root / member.filename).resolve()
                assert destination.is_relative_to(extract_root.resolve()), "Unsafe archive path"
            zipped.extractall(extract_root)
        extraction_marker.write_text(archive_sha256)
    assert any(wiki.glob("wiki-*.jsonl")), "FEVER wiki-page extraction is incomplete."
    command = [PYTHON, "scripts/build_fever_fts.py", "--wiki-pages-dir", str(wiki), "--output", str(INDEX)]
    if MAX_PASSAGES is not None:
        command += ["--max-passages", str(MAX_PASSAGES)]
    subprocess.run(command, check=True, cwd=REPO)
    shutil.copyfile(INDEX, DRIVE_INDEX)
print("Index ready:", INDEX)

## Validate gold, inspect retrieved evidence, and confirm model availability
The coverage export is for human inspection, not an automatic equivalence claim.
It does not alter the annotations or provide gold passages to the generation run.

In [ ]:
RUN_NAME = "pilot_followup_v3"
REASONING_EFFORT = "low"
MIN_COMPLETION_TOKENS = 4096
MODEL = "openai/gpt-oss-120b"  # Choose an ID printed below if unavailable.
inspection = """
from pathlib import Path
import json
from scripts.validate_gold import load_gold
from src.experiments.protocol import load_config, require_segmenter
from src.experiments.runner import pinned_segmenter
from scripts.build_fever_fts import FtsRetriever
print('Checking annotation configuration and sentence model...', flush=True)
config=load_config(Path('configs/evaluation_halueval_colab.yaml'))
with pinned_segmenter(require_segmenter(config)):
    gold=load_gold(Path('data/gold/pilot_dev.jsonl'),'dev')
print('Validating gold annotations succeeded; checking retrieval...', flush=True)
with FtsRetriever(Path('indexes/fever_fts.sqlite')) as retriever:
    rows=[{'record_id':r.id,'claim_id':c.id,'claim':c.text,
           'retrieved':[p.__dict__ for p in retriever.retrieve(c.text,5)]}
          for r in gold for c in r.check_worthy_claims]
Path('results').mkdir(exist_ok=True)
Path('results/pilot_retrieval_review.json').write_text(json.dumps(rows,ensure_ascii=False,indent=2))
print('Reviewed gold:',len(gold),'records;',sum(len(r.claims) for r in gold),'claims')
print('Index passages:',retriever.info['passage_count'])
print('Retrieval review saved; checking Groq credentials and model access...', flush=True)
from groq import Groq
import os
model = os.environ["PILOT_MODEL"]
model_ids={model.id for model in Groq().models.list().data}
print('Available model IDs:', ', '.join(sorted(model_ids)), flush=True)
if model not in model_ids:
    raise ValueError(f'Model {model!r} is unavailable. Set MODEL at the top of this cell to an available text-generation model ID and rerun. The benchmark will use the same selection.')
print('Configured model is available:', model)
"""
inspection_result = subprocess.run(
    [PYTHON, "-u", "-c", inspection], cwd=REPO,
    env={**os.environ, "PILOT_MODEL": MODEL},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(inspection_result.stdout, flush=True)
if inspection_result.returncode:
    raise RuntimeError(
        "Pilot inspection failed. See the Python error printed above. "
        "Fix that error and rerun this cell; the completed index can be reused."
    )
shutil.copyfile(REPO / "results/pilot_retrieval_review.json", DRIVE_ROOT / f"{RUN_NAME}_retrieval_review.json")

## Execute pipeline + B0
Use a new output directory for changed code/settings. Set `RESUME=True` only to
resume exactly the same experiment. The output folder and caches live on Drive.

In [ ]:
RESUME = False
RUN_DIR = DRIVE_ROOT / RUN_NAME
command = [PYTHON, "scripts/run_benchmark.py",
           "--options-file", "configs/pilot_followup_v3.options.json",
           "--reasoning-effort", REASONING_EFFORT,
           "--gptoss-min-completion-tokens", str(MIN_COMPLETION_TOKENS),
           "--config", "configs/evaluation_halueval_colab.yaml",
           "--inputs", "data/gold/pilot_dev.jsonl", "--splits", "data/gold/pilot_splits.json",
           "--provider", "groq", "--model", MODEL,
           "--task", "repair", "--split", "dev", "--output-dir", str(RUN_DIR)]
if RESUME:
    command.append("--resume")
benchmark_result = subprocess.run(
    [command[0], "-u", *command[1:]], cwd=REPO,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(benchmark_result.stdout, flush=True)
BACKUP = Path("/content") / f"{RUN_NAME}_results.zip"
if (RUN_DIR / "manifest.json").exists():
    subprocess.run([PYTHON, "scripts/export_pilot_results.py", "--root", str(DRIVE_ROOT),
                    "--run-name", RUN_NAME, "--output", str(BACKUP)], check=True, cwd=REPO)
    print("Backup created:", BACKUP)
if benchmark_result.returncode:
    errors_path = RUN_DIR / "errors.jsonl"
    if errors_path.is_file():
        print("Saved record errors:", errors_path, flush=True)
        for line in errors_path.read_text().splitlines():
            item = json.loads(line)
            print(json.dumps({key: item.get(key) for key in
                              ("record_id", "system_id", "error")}, indent=2), flush=True)
    raise RuntimeError(
        "Benchmark failed. Inspect the output and saved errors above before retrying. "
        "Keep the run directory; resume requires identical inputs, code, and settings."
    )
print("Execution saved:", RUN_DIR)


## Score reference agreement and export independent repair review
This is the next review stage: humans assess the newly generated repairs, rather
than re-reviewing original gold. A final repair report requires adjudicated output
ratings; reference similarity alone is not factual correctness.

In [ ]:
METRICS = DRIVE_ROOT / f"{RUN_NAME}_reference_metrics.json"
if not METRICS.exists():
    subprocess.run([PYTHON,"scripts/run_eval.py","--run-dir",str(RUN_DIR),
        "--gold","data/gold/pilot_dev.jsonl","--output",str(METRICS)],check=True,cwd=REPO)
PACKET = DRIVE_ROOT / f"{RUN_NAME}_review"
if not PACKET.exists():
    subprocess.run([PYTHON,"scripts/review_repairs.py","export","--run-dir",str(RUN_DIR),
        "--gold","data/gold/pilot_dev.jsonl","--packet-dir",str(PACKET)],check=True,cwd=REPO)
PROGRESS = DRIVE_ROOT / f"{RUN_NAME}_progress"
if not PROGRESS.exists():
    subprocess.run([PYTHON,"scripts/build_evaluation_report.py","--run-dir",str(RUN_DIR),
        "--gold","data/gold/pilot_dev.jsonl","--mode","progress","--output-dir",str(PROGRESS)],check=True,cwd=REPO)
print("Give reviewers only:", PACKET / "reviewer")
print("After independent review/adjudication, follow docs/independent_evaluation.md to build the final pilot report.")
DIAGNOSTICS = DRIVE_ROOT / f"{RUN_NAME}_diagnostics.json"
if not DIAGNOSTICS.exists():
    inspect_command = [PYTHON, "scripts/inspect_pilot_run.py", "--run-dir", str(RUN_DIR),
                       "--gold", "data/gold/pilot_dev.jsonl", "--output", str(DIAGNOSTICS)]
    previous_run = DRIVE_ROOT / "pilot_repair_gptoss_v2"
    if previous_run.exists():
        inspect_command += ["--baseline-run", str(previous_run)]
    subprocess.run(inspect_command, check=True, cwd=REPO)
subprocess.run([PYTHON, "scripts/export_pilot_results.py", "--root", str(DRIVE_ROOT),
                "--run-name", RUN_NAME, "--output", str(BACKUP)], check=True, cwd=REPO)
from google.colab import files
files.download(str(BACKUP))
